In [1]:
import math
import pandas as pd
import numpy as np
import copy
from pathlib import Path
from typing import Any, Dict
import yaml

In [2]:
turbines = pd.read_excel("TurbineList_VN_2030.xlsx", sheet_name="Sheet2")
turbines 

,Model,OEM,Status,T_class_flag,Class,Capacity (MW),Cut-in (m/s),Rated (m/s),Cut-out (m/s),Survival speed,Hub height,Rotor diameter (m),Power curve,Notes / source,
0,V112-3.45,Vestas,Legacy commercial / no longer commercially ava...,0,S / IIA / IIIA,3.45,3.0,13.00,25,59.5,site-/variant-specific,112,Secondary DB available,Secondary turbine DB shows cut-in/rated/cut-ou...,NaN
1,SWT-3.6-107,Siemens (legacy) / Siemens Gamesa,Legacy commercial / no longer commercially ava...,0,IEC IA (secondary),3.60,4.0,16.50,25,70,80 m or site-specific,107,Not public in checked sources,I used the offshore SWT-3.6-107 row: hub heigh...,NaN
2,E-160-EP5,Enercon,Commercial / deployed (onshore) (Enercon),0,II B / III A,5.56,2.5,12.00,28,52.5 (3-s gust),99 / 120 / 160 / 166 m,160,Not public in checked sources,Official Enercon page gives class and hub-heig...,NaN
3,V162-6.2 MW,Vestas,Commercial / serial-production onshore product...,0,IEC S,6.20,3.0,10.50,25,39.5,site-specific / multiple variants,162,Not public in checked sources,"Official Vestas page gives rated power, cut-in...",NaN
4,N163-6.X,Nordex,Commercial / deployed (first installation 2022...,0,IEC S,6.80,3.0,12.50,26,52.5,113–164 m,163,Not public in checked sources,"Secondary DB provides class, operating speeds,...",NaN
5,SG 8.0-167 DD,Siemens Gamesa,Commercial / deployed offshore (Siemens Gamesa),0,I/S (secondary also lists S/IB),8.00,3.0,13.00,28,79.8*,site-specific,167,Secondary DB available; no OEM full binned cur...,Official page gives 8 MW / 167 m / wind class;...,NaN
6,V164-9.5 MW,MHI Vestas / Vestas,"Commercial / deployed offshore, but legacy pla...",0,IEC S,9.50,3.5,14.00,25,NaN,≈105 m,164,Partial / secondary only,Secondary sources give hub height ≈105 m and o...,NaN
7,V174-9.5 MW,MHI Vestas / Vestas,Commercial / deployed offshore (Vestas),1,"IEC Ib,T",9.50,3.0,12.00,31,NaN,≈110 m (reference/prototype value),174,Partial only; I did not locate a full current ...,NaN,NaN
8,IEC-III-10MW Ref,Proxy / user-defined reference,Reference / proxy only,0,IEC III (proxy),10.00,3.0,9.80,25,37.5,130,185,Proxy / user-defined,No single standard public offshore reference u...,NaN
9,Dongfang DEW 10MW-185,Dongfang Electric,Commercial / early commercial Chinese offshore...,0,— (brochure describes high-wind/typhoon-resist...,10.00,3.0,12.00,25,77.0 (3-s),115 m,185,Not public in checked sources,"Dongfang brochure provides operating speeds, r...",NaN


In [3]:
def km_per_deg_lon(lat_deg: float) -> float:
    return 111.320 * math.cos(math.radians(lat_deg))

def km_per_deg_lat(lat_deg: float) -> float:
    return 110.574

def add_km_offset(lat0: float, lon0: float, dx_km: float, dy_km: float):
    """
    Convert local x/y offsets in km to lat/lon degrees.
    dx_km: east-west offset (+ east)
    dy_km: north-south offset (+ north)
    """
    lat = lat0 + dy_km / km_per_deg_lat(lat0)
    lon = lon0 + dx_km / km_per_deg_lon(lat0)
    return lat, lon


# def generate_wombat_layout(
#     lat_c: float,
#     lon_c: float,
#     turbine_rating_mw: float = 15.0,
#     rotor_diameter_m: float = 220.0,
#     cell_size_deg: float = 0.25,
#     n_strings: int = 8,
#     margin_frac: float = 0.10,
#     spacing_along_d: float = 7.0,
#     spacing_cross_d: float = 10.0,
#     max_turbines: int | None = None,
#     max_capacity_mw: float | None = None,
#     substation_id: str = "OSS1",
#     turbine_yaml: str = "enercon_E-126_7_58_EP8.yaml",
#     substation_yaml: str = "osw_substation.yaml",
#     array_yaml: str = "array.yaml",
#     export_yaml: str = "export.yaml",
# ):
#     """
#     Build a WOMBAT layout DataFrame using spacing-based turbine placement.

#     Assumptions:
#     - cell center is (lat_c, lon_c)
#     - one OSS at the center
#     - turbines arranged on a regular grid inside the cell
#     - turbine spacing is derived from rotor diameter
#     - strings are assigned by angular sector around the OSS
#     - order is assigned by distance from OSS within each string

#     Parameters
#     ----------
#     lat_c, lon_c : float
#         Cell center coordinates in WGS-84.
#     turbine_rating_mw : float
#         Turbine rated power in MW.
#     rotor_diameter_m : float
#         Rotor diameter in meters.
#     cell_size_deg : float
#         Cell size in degrees (assumed square).
#     n_strings : int
#         Number of feeder strings.
#     margin_frac : float
#         Fraction of cell width/height reserved as edge buffer on each side.
#     spacing_along_d : float
#         Turbine spacing multiplier in rotor diameters in one direction.
#     spacing_cross_d : float
#         Turbine spacing multiplier in rotor diameters in the other direction.
#     max_turbines : int | None
#         Optional cap on turbine count. If None and max_capacity_mw is provided,
#         calculated from max_capacity_mw / turbine_rating_mw.
#     max_capacity_mw : float | None
#         Optional cap on project capacity in MW. If None and max_turbines is provided,
#         calculated from max_turbines * turbine_rating_mw.
#     """

#     if not (0 <= margin_frac < 0.5):
#         raise ValueError("margin_frac must be in [0, 0.5).")
#     if rotor_diameter_m <= 0:
#         raise ValueError("rotor_diameter_m must be > 0.")
#     if turbine_rating_mw <= 0:
#         raise ValueError("turbine_rating_mw must be > 0.")
#     if n_strings < 1:
#         raise ValueError("n_strings must be >= 1.")
#     if spacing_along_d <= 0 or spacing_cross_d <= 0:
#         raise ValueError("spacing multipliers must be > 0.")

#     # Derive max_turbines or max_capacity_mw if one is provided but not the other
#     if max_capacity_mw is not None and max_turbines is None:
#         max_turbines = int(max_capacity_mw // turbine_rating_mw)
#     elif max_turbines is not None and max_capacity_mw is None:
#         max_capacity_mw = max_turbines * turbine_rating_mw

#     # Cell dimensions and area
#     width_km = km_per_deg_lon(lat_c) * cell_size_deg
#     height_km = km_per_deg_lat(lat_c) * cell_size_deg
#     area_km2 = width_km * height_km

#     # Apply edge buffer
#     usable_width_km = width_km * (1 - 2 * margin_frac)
#     usable_height_km = height_km * (1 - 2 * margin_frac)

#     if usable_width_km <= 0 or usable_height_km <= 0:
#         raise ValueError("Usable area is zero or negative. Reduce margin_frac.")

#     # Spacing-based layout from rotor diameter
#     d_km = rotor_diameter_m / 1000.0
#     spacing_x_km = spacing_along_d * d_km
#     spacing_y_km = spacing_cross_d * d_km

#     # Number of turbines that fit in each direction
#     # +1 because if usable width equals spacing, two points can fit at both ends
#     n_cols_fit = max(1, int(math.floor(usable_width_km / spacing_x_km)) + 1)
#     n_rows_fit = max(1, int(math.floor(usable_height_km / spacing_y_km)) + 1)

#     n_turbines_theoretical = n_cols_fit * n_rows_fit
#     theoretical_capacity_mw = n_turbines_theoretical * turbine_rating_mw

#     # Apply project caps
#     n_turbines = n_turbines_theoretical

#     if max_capacity_mw is not None:
#         n_turbines = min(n_turbines, int(max_capacity_mw // turbine_rating_mw))

#     if max_turbines is not None:
#         n_turbines = min(n_turbines, max_turbines)

#     if n_turbines < 1:
#         raise ValueError(
#             "Computed n_turbines < 1 after spacing/caps. "
#             "Check rotor diameter, spacing, cell size, or caps."
#         )

#     actual_capacity_mw = n_turbines * turbine_rating_mw

#     # Determine actual grid shape after capping
#     n_cols = min(n_cols_fit, math.ceil(math.sqrt(n_turbines)))
#     n_rows = math.ceil(n_turbines / n_cols)

#     while n_rows > n_rows_fit and n_cols < n_cols_fit:
#         n_cols += 1
#         n_rows = math.ceil(n_turbines / n_cols)

#     if n_rows > n_rows_fit:
#         raise ValueError(
#             "Capped turbine count still does not fit in the available spacing-based grid."
#         )

#     # Actual occupied footprint for the chosen grid
#     occupied_width_km = spacing_x_km * max(n_cols - 1, 0)
#     occupied_height_km = spacing_y_km * max(n_rows - 1, 0)

#     # Center the turbine grid on the OSS
#     x0 = -occupied_width_km / 2.0
#     y0 = -occupied_height_km / 2.0

#     # Place OSS at cell centroid
#     rows = [{
#         "id": substation_id,
#         "substation_id": substation_id,
#         "name": substation_id,
#         "type": "substation",
#         "longitude": lon_c,
#         "latitude": lat_c,
#         "string": "",
#         "order": "",
#         "distance": "",
#         "subassembly": substation_yaml,
#         "upstream_cable": export_yaml,
#     }]

#     # Create turbine candidate points
#     turbines = []
#     t = 0
#     for r in range(n_rows):
#         for c in range(n_cols):
#             if t >= n_turbines:
#                 break
#             c_eff = c if r % 2 == 0 else (n_cols - 1 - c)

#             x = x0 + c_eff * spacing_x_km
#             y = y0 + r * spacing_y_km
#             lat, lon = add_km_offset(lat_c, lon_c, x, y)

#             turbines.append({
#                 "tmp_index": t,
#                 "x_km": x,
#                 "y_km": y,
#                 "latitude": lat,
#                 "longitude": lon,
#             })
#             t += 1

#     # Assign strings by angular sector around OSS
#     for turb in turbines:
#         angle = math.atan2(turb["y_km"], turb["x_km"])  # -pi to pi
#         angle_0_2pi = angle if angle >= 0 else angle + 2 * math.pi
#         turb["string"] = min(
#             int(angle_0_2pi / (2 * math.pi) * n_strings),
#             n_strings - 1
#         )
#         turb["radius"] = math.hypot(turb["x_km"], turb["y_km"])

#     # Within each string, sort by distance from OSS and assign order
#     grouped = {}
#     for turb in turbines:
#         grouped.setdefault(turb["string"], []).append(turb)
#     global_turbine_idx = 1
#     final_turbines = []

#     for s in range(n_strings):
#         group = grouped.get(s, [])
#         group = sorted(group, key=lambda z: z["radius"])

#         for order, turb in enumerate(group):
#             tid = f"T{global_turbine_idx}"
#             final_turbines.append({
#                 "id": tid,
#                 "substation_id": substation_id,
#                 "name": tid,
#                 "type": "turbine",
#                 "longitude": turb["longitude"],
#                 "latitude": turb["latitude"],
#                 "string": s,
#                 "order": order,
#                 "distance": 0,
#                 "subassembly": turbine_yaml,
#                 "upstream_cable": array_yaml,
#             })
#             global_turbine_idx += 1

#     # Combine
#     layout = pd.DataFrame(rows + final_turbines)

#     # Implied density metrics
#     occupied_area_km2 = max(occupied_width_km, spacing_x_km) * max(occupied_height_km, spacing_y_km)
#     occupied_density_mw_per_km2 = (
#         actual_capacity_mw / occupied_area_km2 if occupied_area_km2 > 0 else float("nan")
#     )
#     cell_density_mw_per_km2 = actual_capacity_mw / area_km2 if area_km2 > 0 else float("nan")

#     meta = {
#         "lat_center": lat_c,
#         "lon_center": lon_c,
#         "cell_width_km": width_km,
#         "cell_height_km": height_km,
#         "cell_area_km2": area_km2,
#         "usable_width_km": usable_width_km,
#         "usable_height_km": usable_height_km,
#         "margin_frac": margin_frac,
#         "rotor_diameter_m": rotor_diameter_m,
#         "spacing_along_d": spacing_along_d,
#         "spacing_cross_d": spacing_cross_d,
#         "spacing_x_km": spacing_x_km,
#         "spacing_y_km": spacing_y_km,
#         "n_cols_fit": n_cols_fit,
#         "n_rows_fit": n_rows_fit,
#         "n_turbines_theoretical": n_turbines_theoretical,
#         "theoretical_capacity_mw": theoretical_capacity_mw,
#         "n_turbines": n_turbines,
#         "turbine_rating_mw": turbine_rating_mw,
#         "actual_capacity_mw": actual_capacity_mw,
#         "max_turbines": max_turbines,
#         "max_capacity_mw": max_capacity_mw,
#         "n_cols_used": n_cols,
#         "n_rows_used": n_rows,
#         "occupied_width_km": occupied_width_km,
#         "occupied_height_km": occupied_height_km,
#         "occupied_density_mw_per_km2": occupied_density_mw_per_km2,
#         "cell_density_mw_per_km2": cell_density_mw_per_km2,
#         "n_strings": n_strings,
#     }

#     return layout, meta

In [8]:
def generate_wombat_layout(
    lat_c: float,
    lon_c: float,
    turbine_rating_mw: float = 15.0,
    rotor_diameter_m: float = 220.0,
    cell_size_deg: float = 0.25,
    margin_frac: float = 0.10,
    spacing_along_d: float = 7.0,
    spacing_cross_d: float = 10.0,
    max_turbines: int | None = None,
    max_capacity_mw: float | None = None,
    substation_id: str = "OSS1",
    turbine_yaml: str = "vestas_v90_failure_200.yaml",
    substation_yaml: str = "offshore_substation.yaml",
    array_yaml: str = "array.yaml",
    export_yaml: str = "export.yaml",
):
    """
    Build a WOMBAT layout DataFrame using spacing-based turbine placement.

    Assumptions:
    - cell center is (lat_c, lon_c)
    - one OSS at the center
    - turbines arranged on a regular grid inside the cell
    - turbine spacing is derived from rotor diameter
    - all turbines are assigned to a single string
    - order is assigned by distance from OSS
    """

    if not (0 <= margin_frac < 0.5):
        raise ValueError("margin_frac must be in [0, 0.5).")
    if rotor_diameter_m <= 0:
        raise ValueError("rotor_diameter_m must be > 0.")
    if turbine_rating_mw <= 0:
        raise ValueError("turbine_rating_mw must be > 0.")
    if spacing_along_d <= 0 or spacing_cross_d <= 0:
        raise ValueError("spacing multipliers must be > 0.")

    # Derive max_turbines or max_capacity_mw if one is provided but not the other
    if max_capacity_mw is not None and max_turbines is None:
        max_turbines = int(max_capacity_mw // turbine_rating_mw)
    elif max_turbines is not None and max_capacity_mw is None:
        max_capacity_mw = max_turbines * turbine_rating_mw

    # Cell dimensions and area
    width_km = km_per_deg_lon(lat_c) * cell_size_deg
    height_km = km_per_deg_lat(lat_c) * cell_size_deg

    # Apply edge buffer
    usable_width_km = width_km * (1 - 2 * margin_frac)
    usable_height_km = height_km * (1 - 2 * margin_frac)

    if usable_width_km <= 0 or usable_height_km <= 0:
        raise ValueError("Usable area is zero or negative. Reduce margin_frac.")

    # Spacing-based layout from rotor diameter
    d_km = rotor_diameter_m / 1000.0
    spacing_x_km = spacing_along_d * d_km
    spacing_y_km = spacing_cross_d * d_km

    # Number of turbines that fit in each direction
    n_cols_fit = max(1, int(math.floor(usable_width_km / spacing_x_km)) + 1)
    n_rows_fit = max(1, int(math.floor(usable_height_km / spacing_y_km)) + 1)

    n_turbines_theoretical = n_cols_fit * n_rows_fit

    # Apply project caps
    n_turbines = n_turbines_theoretical

    if max_capacity_mw is not None:
        n_turbines = min(n_turbines, int(max_capacity_mw // turbine_rating_mw))

    if max_turbines is not None:
        n_turbines = min(n_turbines, max_turbines)

    if n_turbines < 1:
        raise ValueError(
            "Computed n_turbines < 1 after spacing/caps. "
            "Check rotor diameter, spacing, cell size, or caps."
        )

    # Determine actual grid shape after capping
    n_cols = min(n_cols_fit, math.ceil(math.sqrt(n_turbines)))
    n_rows = math.ceil(n_turbines / n_cols)

    while n_rows > n_rows_fit and n_cols < n_cols_fit:
        n_cols += 1
        n_rows = math.ceil(n_turbines / n_cols)

    if n_rows > n_rows_fit:
        raise ValueError(
            "Capped turbine count still does not fit in the available spacing-based grid."
        )

    # Actual occupied footprint for the chosen grid
    x0 = -((n_cols - 1) * spacing_x_km) / 2.0
    y0 = -((n_rows - 1) * spacing_y_km) / 2.0

    # Place OSS at cell centroid
    rows = [{
        "id": substation_id,
        "substation_id": substation_id,
        "name": substation_id,
        "type": "substation",
        "longitude": lon_c,
        "latitude": lat_c,
        "string": "",
        "order": "",
        "distance": "",
        "subassembly": substation_yaml,
        "upstream_cable": export_yaml,
    }]

    # Create turbine candidate points
    turbines = []
    t = 0
    for r in range(n_rows):
        for c in range(n_cols):
            if t >= n_turbines:
                break

            x = x0 + c * spacing_x_km
            y = y0 + r * spacing_y_km
            lat, lon = add_km_offset(lat_c, lon_c, x, y)

            turbines.append({
                "id": f"S00T{t + 1}",
                "substation_id": substation_id,
                "name": f"S00T{t + 1}",
                "type": "turbine",
                "longitude": lon,
                "latitude": lat,
                "string": 0,
                "order": t,
                "distance": 0,
                "subassembly": turbine_yaml,
                "upstream_cable": array_yaml,
            })
            t += 1

    # Combine
    layout = pd.DataFrame(rows + turbines)
    meta = {
        "n_turbines": n_turbines,
        "n_strings": 1,
    }

    return layout, meta

In [9]:
for _, row in turbines.iterrows():
    # Extract turbine-specific details
    turbine_model = row["Model"]
    turbine_rating_mw = row["Capacity (MW)"]
    rotor_diameter_m = row["Rotor diameter (m)"]

    turbine_yaml_file = f"{turbine_model}.yaml"
    turbine_yaml_path = Path(f"library/turbines/{turbine_yaml_file}")
    if not turbine_yaml_path.exists():
        print(f"YAML file for {turbine_model} not found. Skipping...")
        continue

    with open(turbine_yaml_path, "r") as yaml_file:
        turbine_config = yaml.safe_load(yaml_file)

    # Generate layout using the existing function
    try:
        layout, meta = generate_wombat_layout(
            lat_c=8.75,
            lon_c=107.5,
            turbine_rating_mw=turbine_rating_mw,
            rotor_diameter_m=rotor_diameter_m,
            cell_size_deg=0.25,
            margin_frac=0.10,
            spacing_along_d=7.0,
            spacing_cross_d=10.0,
            max_turbines=80,
            max_capacity_mw=800,
        )
    except ValueError as e:
        print(f"Error generating layout for {turbine_model}: {e}")
        continue

    # Extract substation coordinates from the generated layout
    substation_row = layout.iloc[0].to_dict()
    substation_row.update({
        "subassembly": "osw_substation.yaml",
        "upstream_cable": "export.yaml"
    })

    # Update layout DataFrame
    layout = pd.concat([pd.DataFrame([substation_row]), layout.iloc[1:]], ignore_index=True)

    layout.loc[1:, "subassembly"] = turbine_yaml_file
    layout.loc[1:, "upstream_cable"] = "array.yaml"

    # Save the layout to a CSV file
    layout_file = Path(f"library/project/plant/layout_{turbine_model}.csv")
    layout.to_csv(layout_file, index=False)

    # substation_file = Path("library/substations/osw_substation.yaml")
    # substation_data = {
    #     "model": turbine_model,
    #     "layout_file": str(layout_file),
    #     "turbine_config": turbine_config,
    # }
    # with open(substation_file, "w") as sub_file:
    #     yaml.safe_dump(substation_data, sub_file)

    print(f"Layout and substation file for {turbine_model} generated.")

Layout and substation file for V112-3.45 generated.
Layout and substation file for SWT-3.6-107 generated.
Layout and substation file for E-160-EP5 generated.
Layout and substation file for V162-6.2 MW generated.
Layout and substation file for N163-6.X generated.
Layout and substation file for SG 8.0-167 DD generated.
Layout and substation file for V164-9.5 MW generated.
Layout and substation file for V174-9.5 MW generated.
Layout and substation file for IEC-III-10MW Ref generated.
Layout and substation file for Dongfang DEW 10MW-185 generated.
Layout and substation file for SG 11.0-200 DD generated.
Layout and substation file for Haliade-X 12 MW generated.
Layout and substation file for IEC-II-12MW Ref generated.
Layout and substation file for MySE12-242 generated.
Layout and substation file for Haliade-X 14 MW generated.
Layout and substation file for SG 14-222 DD generated.
Layout and substation file for SG 14-236 DD generated.
Layout and substation file for IEA 15 MW Reference gener

In [ ]:
def interpolate_power_curve_single(
    wind_speeds: np.ndarray,
    cut_in: float,
    rated_ws: float,
    cut_out: float,
    rated_power_mw: float,
) -> np.ndarray:
    """
    Build a simple turbine power curve using a piecewise rule:

    - ws < cut_in                -> 0
    - cut_in <= ws < rated_ws    -> cubic interpolation
    - rated_ws <= ws <= cut_out  -> rated power
    - ws > cut_out               -> 0

    The below-rated interpolation is:
        P = Pr * (ws^3 - cut_in^3) / (rated_ws^3 - cut_in^3)
    """
    ws = np.asarray(wind_speeds, dtype=float)
    power_kw = np.zeros_like(ws, dtype=float)

    if not (cut_in < rated_ws <= cut_out):
        raise ValueError(
            f"Invalid speed thresholds: cut_in={cut_in}, rated_ws={rated_ws}, cut_out={cut_out}"
        )
    if rated_power_mw <= 0:
        raise ValueError(f"rated_power_mw must be > 0, got {rated_power_mw}")

    rated_power_kw = rated_power_mw * 1000.0

    # Region II: below-rated cubic ramp
    ramp_mask = (ws >= cut_in) & (ws < rated_ws)
    if np.any(ramp_mask):
        denom = rated_ws**3 - cut_in**3
        if denom <= 0:
            raise ValueError(
                f"Invalid cubic interpolation denominator for cut_in={cut_in}, rated_ws={rated_ws}"
            )
        power_kw[ramp_mask] = rated_power_kw * (
            (ws[ramp_mask]**3 - cut_in**3) / denom
        )

    # Region III: rated plateau
    rated_mask = (ws >= rated_ws) & (ws <= cut_out)
    power_kw[rated_mask] = rated_power_kw

    # Region I and IV remain zero
    power_kw = np.clip(power_kw, 0, rated_power_kw)
    return power_kw


def build_power_curves_for_turbines(
    turbine_df: pd.DataFrame,
    ws_min: float = 0.0,
    ws_max: float = 35.0,
    ws_step: float = 0.5,
    return_long_format: bool = False,
) -> pd.DataFrame:
    """
    Generate power curves for multiple turbines with consistent 0.5 m/s bin width.
    Ensures only one power_kw value per wind speed.
    """
    required_cols = [
        "Model",
        "Capacity (MW)",
        "Cut-in (m/s)",
        "Rated (m/s)",
        "Cut-out (m/s)",
    ]
    missing = [c for c in required_cols if c not in turbine_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Generate wind speeds with exact 0.5 bin width
    wind_speeds = np.round(np.arange(ws_min, ws_max + ws_step / 2, ws_step), 1)

    records = []
    curve_arrays = []

    for _, row in turbine_df.iterrows():
        cut_in = float(row["Cut-in (m/s)"])
        rated_ws = float(row["Rated (m/s)"])
        cut_out = float(row["Cut-out (m/s)"])
        rated_power_mw = float(row["Capacity (MW)"])

        power_curve_kw = interpolate_power_curve_single(
            wind_speeds=wind_speeds,
            cut_in=cut_in,
            rated_ws=rated_ws,
            cut_out=cut_out,
            rated_power_mw=rated_power_mw,
        )

        if return_long_format:
            for ws, p_kw in zip(wind_speeds, power_curve_kw):
                records.append({
                    "Model": row["Model"],
                    "Capacity (MW)": rated_power_mw,
                    "Cut-in (m/s)": cut_in,
                    "Rated (m/s)": rated_ws,
                    "Cut-out (m/s)": cut_out,
                    "wind_speed_ms": ws,
                    "power_kw": p_kw,
                })
        else:
            curve_arrays.append(power_curve_kw)

    if return_long_format:
        return pd.DataFrame(records)

    out = turbine_df.copy()
    out["wind_speeds_ms"] = [wind_speeds] * len(out)
    out["power_curve_kw"] = curve_arrays
    return out

In [4]:
# Long format
power_curve_long = build_power_curves_for_turbines(
    turbines,
    ws_min=0,
    ws_max=35,
    ws_step=0.5,
    return_long_format=True,
)

In [5]:
power_curve_long.head(15)

,Model,Capacity (MW),Cut-in (m/s),Rated (m/s),Cut-out (m/s),wind_speed_ms,power_kw
0,V112-3.45,3.45,3.0,13.0,25.0,0.0,0.000000
1,V112-3.45,3.45,3.0,13.0,25.0,0.5,0.000000
2,V112-3.45,3.45,3.0,13.0,25.0,1.0,0.000000
3,V112-3.45,3.45,3.0,13.0,25.0,1.5,0.000000
4,V112-3.45,3.45,3.0,13.0,25.0,2.0,0.000000
5,V112-3.45,3.45,3.0,13.0,25.0,2.5,0.000000
6,V112-3.45,3.45,3.0,13.0,25.0,3.0,0.000000
7,V112-3.45,3.45,3.0,13.0,25.0,3.5,25.239055
8,V112-3.45,3.45,3.0,13.0,25.0,4.0,58.824885
9,V112-3.45,3.45,3.0,13.0,25.0,4.5,101.949885


In [15]:
def export_power_curves_to_csv(power_curve_df: pd.DataFrame, output_dir: str = "."):
    import os
    os.makedirs(output_dir, exist_ok=True)

    for model, group in power_curve_df.groupby("Model"):
        df_clean = pd.DataFrame({
            "windspeed_ms": group["wind_speed_ms"].values,
            "power_kw": group["power_kw"].values
        })
        
        df_clean = df_clean[df_clean["power_kw"] > 0].copy()
        df_dedup = df_clean.groupby("windspeed_ms", as_index=False)["power_kw"].mean()
        df_dedup["power_kw"] = df_dedup["power_kw"].round(0).astype(int)
        df_dedup = df_dedup.sort_values("windspeed_ms").reset_index(drop=True)
        output_file = os.path.join(output_dir, f"{model}.csv")
        df_dedup.to_csv(output_file, index=False)


In [16]:
export_power_curves_to_csv(power_curve_long, output_dir="library/turbines")

In [ ]:
# Gen config files for turbines
FIXED_CAPEX_KW = 1300

# CAPEX fractions for each maintenance/failure event
# These fractions are now defined based on realistic assumptions.
MATERIAL_FRACTIONS = {
    "blade": {
        "maintenance": {
            "annual blade inspection": 0.0001,
        },
        "failures": {
            "blade minor repair": 0.0005,
            "blade major repair": 0.002,
        },
    },
    "pitch_system": {
        "maintenance": {
            "annual pitch service": 0.0002,
        },
        "failures": {
            "pitch actuator repair": 0.0003,
            "pitch system major repair": 0.001,
        },
    },
    "yaw_system": {
        "maintenance": {
            "annual yaw service": 0.00015,
        },
        "failures": {
            "yaw drive repair": 0.00025,
        },
    },
    "drivetrain": {
        "maintenance": {
            "annual drivetrain inspection": 0.0004,
        },
        "failures": {
            "drivetrain minor repair": 0.001,
            "drivetrain major repair": 0.005,
            "drivetrain replacement": 0.01,
        },
    },
    "generator": {
        "maintenance": {
            "annual generator inspection": 0.0003,
        },
        "failures": {
            "generator repair": 0.0008,
            "generator replacement": 0.007,
        },
    },
    "converter": {
        "maintenance": {
            "annual converter inspection": 0.0001,
        },
        "failures": {
            "converter repair": 0.0004,
        },
    },
    "transformer": {
        "maintenance": {
            "annual transformer inspection": 0.0002,
        },
        "failures": {
            "transformer major repair": 0.003,
        },
    },
}


BASE_TEMPLATE: Dict[str, Any] = {
    "capacity_kw": None,
    "capex_kw": None,
    "power_curve": {
        "file": None,
        "bin_width": 0.5,
    },
    "blade": {
        "name": "blade",
        "maintenance": [
            {
                "description": "annual blade inspection",
                "time": 8,
                "materials": None,
                "service_equipment": "CTV",
                "frequency": 365,
            }
        ],
        "failures": [
            {
                "scale": 20,
                "shape": 1,
                "time": 10,
                "materials": None,
                "service_equipment": "CTV",
                "operation_reduction": 0,
                "level": 2,
                "description": "blade minor repair",
            },
            {
                "scale": 50,
                "shape": 1,
                "time": 30,
                "materials": None,
                "service_equipment": "SCN",
                "operation_reduction": 0,
                "level": 4,
                "description": "blade major repair",
            },
        ],
    },
    "pitch_system": {
        "name": "pitch_system",
        "maintenance": [
            {
                "description": "annual pitch service",
                "time": 6,
                "materials": None,
                "service_equipment": "CTV",
                "frequency": 365,
            }
        ],
        "failures": [
            {
                "scale": 0.886,
                "shape": 1,
                "time": 8,
                "materials": None,
                "service_equipment": "CTV",
                "operation_reduction": 0,
                "level": 2,
                "description": "pitch actuator repair",
            },
            {
                "scale": 2.658,
                "shape": 1,
                "time": 18,
                "materials": None,
                "service_equipment": "SCN",
                "operation_reduction": 0,
                "level": 3,
                "description": "pitch system major repair",
            },
        ],
    },
    "yaw_system": {
        "name": "yaw_system",
        "maintenance": [
            {
                "description": "annual yaw service",
                "time": 5,
                "materials": None,
                "service_equipment": "CTV",
                "frequency": 365,
            }
        ],
        "failures": [
            {
                "scale": 6,
                "shape": 1,
                "time": 7,
                "materials": None,
                "service_equipment": "CTV",
                "operation_reduction": 0,
                "level": 2,
                "description": "yaw drive repair",
            }
        ],
    },
    "drivetrain": {
        "name": "drivetrain",
        "maintenance": [
            {
                "description": "annual drivetrain inspection",
                "time": 10,
                "materials": None,
                "service_equipment": "CTV",
                "frequency": 365,
            }
        ],
        "failures": [
            {
                "scale": 5.530,
                "shape": 1,
                "time": 16,
                "materials": None,
                "service_equipment": "CTV",
                "operation_reduction": 0,
                "level": 3,
                "description": "drivetrain minor repair",
            },
            {
                "scale": 17.280,
                "shape": 1,
                "time": 40,
                "materials": None,
                "service_equipment": "SCN",
                "operation_reduction": 0,
                "level": 4,
                "description": "drivetrain major repair",
            },
            {
                "scale": 27.648,
                "shape": 1,
                "time": 72,
                "materials": None,
                "service_equipment": "LCN",
                "operation_reduction": 0,
                "replacement": True,
                "level": 5,
                "description": "drivetrain replacement",
            },
        ],
    },
    "generator": {
        "name": "generator",
        "maintenance": [
            {
                "description": "annual generator inspection",
                "time": 8,
                "materials": None,
                "service_equipment": "CTV",
                "frequency": 365,
            }
        ],
        "failures": [
            {
                "scale": 16.455,
                "shape": 1,
                "time": 14,
                "materials": None,
                "service_equipment": "CTV",
                "operation_reduction": 0,
                "level": 3,
                "description": "generator repair",
            },
            {
                "scale": 49.366,
                "shape": 1,
                "time": 48,
                "materials": None,
                "service_equipment": "LCN",
                "operation_reduction": 0,
                "replacement": True,
                "level": 5,
                "description": "generator replacement",
            },
        ],
    },
    "converter": {
        "name": "converter",
        "maintenance": [
            {
                "description": "annual converter inspection",
                "time": 4,
                "materials": None,
                "service_equipment": "CTV",
                "frequency": 365,
            }
        ],
        "failures": [
            {
                "scale": 5.577,
                "shape": 1,
                "time": 6,
                "materials": None,
                "service_equipment": "CTV",
                "operation_reduction": 0,
                "level": 2,
                "description": "converter repair",
            }
        ],
    },
    # "transformer": {
    #     "name": "transformer",
    #     "maintenance": [
    #         {
    #             "description": "annual transformer inspection",
    #             "time": 6,
    #             "materials": None,
    #             "service_equipment": "CTV",
    #             "frequency": 365,
    #         }
    #     ],
    #     "failures": [
    #         {
    #             "scale": 19.317,
    #             "shape": 1,
    #             "time": 20,
    #             "materials": None,
    #             "service_equipment": "SCN",
    #             "operation_reduction": 0.50,
    #             "level": 4,
    #             "description": "transformer major repair",
    #         }
    #     ],
    # },
}


def sanitize_filename(name: str) -> str:
    safe = str(name).strip()
    for ch in ['/', '\\', ':', '*', '?', '"', '<', '>', '|']:
        safe = safe.replace(ch, "_")
    return safe


def apply_material_costs_from_total_capex(config: Dict[str, Any], total_capex: float) -> None:
    """
    Fill all maintenance/failure materials directly from total_capex using
    fixed CAPEX fractions. Fractions are defined in MATERIAL_FRACTIONS.
    """
    for component_name, component_cfg in config.items():
        if component_name not in MATERIAL_FRACTIONS:
            continue

        fractions = MATERIAL_FRACTIONS[component_name]

        if "maintenance" in component_cfg:
            for item in component_cfg["maintenance"]:
                desc = item["description"]
                frac = fractions["maintenance"].get(desc, 0)
                item["materials"] = int(total_capex * frac) if frac else None

        if "failures" in component_cfg:
            for item in component_cfg["failures"]:
                desc = item["description"]
                frac = fractions["failures"].get(desc, 0)
                item["materials"] = int(total_capex * frac) if frac else None


def make_turbine_config(
    row: pd.Series,
    model_col: str = "Model",
    capacity_mw_col: str = "Capacity (MW)",
) -> Dict[str, Any]:
    """
    Build one turbine config dict from a dataframe row.

    Rules:
    - capacity_kw = Capacity (MW) * 1000
    - capex_kw = FIXED_CAPEX_KW
    - total_capex = capacity_kw * capex_kw
    - power_curve.file = <Model>.csv
    - failure rates/scales stay fixed
    - materials are computed directly from total_capex using fixed fractions
    """
    model = str(row[model_col]).strip()
    capacity_kw = int(float(row[capacity_mw_col]) * 1000.0)
    capex_kw = FIXED_CAPEX_KW
    total_capex = capacity_kw * capex_kw

    config = copy.deepcopy(BASE_TEMPLATE)
    config["capacity_kw"] = capacity_kw
    config["capex_kw"] = capex_kw
    config["power_curve"] = {
        "file": f"{model}.csv",
        "bin_width": 0.5,
    }

    apply_material_costs_from_total_capex(config, total_capex)
    return config


def write_turbine_configs(
    turbines_df: pd.DataFrame,
    output_dir: str | Path = "library/turbines",
    model_col: str = "Model",
    capacity_mw_col: str = "Capacity (MW)",
    yaml_ext: str = ".yaml",
) -> None:
    """
    Generate one yaml config per turbine and write into library/turbines.
    """
    required_cols = [model_col, capacity_mw_col]
    missing = [c for c in required_cols if c not in turbines_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for _, row in turbines_df.iterrows():
        config = make_turbine_config(
            row=row,
            model_col=model_col,
            capacity_mw_col=capacity_mw_col,
        )

        model = str(row[model_col]).strip()
        yaml_name = sanitize_filename(model) + yaml_ext
        yaml_path = output_dir / yaml_name

        with open(yaml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(
                config,
                f,
                sort_keys=False,
                allow_unicode=True,
                default_flow_style=False,
            )

In [ ]:
write_turbine_configs(
    turbines_df=turbines,
    output_dir="library/turbines",
    model_col="Model",
    capacity_mw_col="Capacity (MW)",
)


In [4]:
def generate_config_files(turbine_names, output_dir="library/project/config"):
    """
    Generate configuration files for each turbine layout.

    Parameters:
    - turbine_names: List of turbine names).
    - output_dir: Directory to save the configuration files.
    """
    base_config = {
        "name": "base_fixed_bottom",
        "weather": "weather.csv",
        "service_equipment": [
            "ctv.yaml",
            "cab.yaml",
            "hlv.yaml",
            "dsv.yaml"
        ],
        "layout": "layout.csv",
        "inflation_rate": 0,
        "fixed_costs": "fixed_costs.yaml",
        "workday_start": 7,
        "workday_end": 19,
        "start_year": 2002,
        "end_year": 2021,
        "site_distance": 116,
        "port": "base_port.yaml"
    }

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for turbine_name in turbine_names:
        layout_file = Path(f"library/project/plant/layout_{turbine_name}.csv")
        if not layout_file.exists():
            print(f"Layout file for {turbine_name} not found. Skipping...")
            continue
        layout_df = pd.read_csv(layout_file)
        
        # Calculate project capacity by reading capacity from each turbine in layout
        turbine_rows = layout_df[layout_df["type"] == "turbine"]
        total_capacity_kw = 0
        
        for _, turb_row in turbine_rows.iterrows():
            # Get the turbine subassembly file (should be the turbine YAML file)
            turbine_yaml = turb_row.get("subassembly", f"{turbine_name}.yaml")
            turbine_yaml_path = Path(f"library/turbines/{turbine_yaml}")
            
            if turbine_yaml_path.exists():
                with open(turbine_yaml_path, "r") as f:
                    turb_config = yaml.safe_load(f)
                    capacity_kw = turb_config.get("capacity_kw", 0)
                    total_capacity_kw += capacity_kw
        
        # Convert to MW (keep full precision - do not round)
        project_capacity_mw = total_capacity_kw / 1000.0

        config = base_config.copy()
        config["name"] = f"{turbine_name}_base"
        config["layout"] = layout_file.name
        config["project_capacity"] = project_capacity_mw

        config_file = output_dir / f"{turbine_name}_base.yaml"
        with open(config_file, "w") as file:
            yaml.safe_dump(config, file, sort_keys=False, default_flow_style=False)

        print(f"Config file generated: {config_file}")

In [5]:
generate_config_files(turbines["Model"].tolist(), output_dir="library/project/config")

Config file generated: library/project/config/V112-3.45_base.yaml
Config file generated: library/project/config/SWT-3.6-107_base.yaml
Config file generated: library/project/config/E-160-EP5_base.yaml
Config file generated: library/project/config/V162-6.2 MW_base.yaml
Config file generated: library/project/config/N163-6.X_base.yaml
Config file generated: library/project/config/SG 8.0-167 DD_base.yaml
Config file generated: library/project/config/V164-9.5 MW_base.yaml
Config file generated: library/project/config/V174-9.5 MW_base.yaml
Config file generated: library/project/config/IEC-III-10MW Ref_base.yaml
Config file generated: library/project/config/Dongfang DEW 10MW-185_base.yaml
Config file generated: library/project/config/SG 11.0-200 DD_base.yaml
Config file generated: library/project/config/Haliade-X 12 MW_base.yaml
Config file generated: library/project/config/IEC-II-12MW Ref_base.yaml
Config file generated: library/project/config/MySE12-242_base.yaml
Config file generated: libra